# 02 Clean: build the municipality-year panel
Reads the `raw_*` tables from notebook 01 and writes `clean_*` tables. Decisions applied (see the README's Data decisions):
1. **Province codes:** KZ → KZN and LIM → LP (one code per province).
2. **Blank irregular-expenditure amounts** → 0 with a `was_blank` flag (decision 7).
3. **Missing audited statements** → kept as flags `has_audited_*`, not dropped or imputed (decision 5).
4. **No rows deleted:** all years kept; `mscoa_era` marks 2018+ for analyses that need consistent financials (decision 8).

The last cell runs reconciliation checks: every check must print PASS.

In [ ]:
from pyspark.sql import functions as F, Window

raw = {t: spark.table(f"raw_{t}") for t in
       ["municipalities", "audit_opinions", "uifwexp", "incexp_v2", "financial_position_v2", "aged_debtor_v2"]}
for t, df in raw.items():
    print(f"raw_{t}: {df.count():,} rows")

## 1. Municipalities: one code per province

In [ ]:
PROVINCE_FIX = {"KZ": "KZN", "LIM": "LP"}
fix = F.create_map(*[F.lit(x) for kv in PROVINCE_FIX.items() for x in kv])

clean_muni = (raw["municipalities"]
    .select(F.col("municipality_demarcation_code").alias("demarcation_code"),
            F.col("municipality_name").alias("name"),
            F.coalesce(fix[F.col("municipality_province_code")], F.col("municipality_province_code")).alias("province_code"),
            F.col("municipality_province_name").alias("province_name_raw"),
            F.col("municipality_category").alias("category"),       # A metro, B local, C district
            F.col("municipality_miif_category").alias("miif_category"),
            F.col("municipality_parent_code").alias("parent_code")))
# one spelling per province name too ("Kwazulu-Natal" vs "KwaZulu-Natal")
NAMES = {"EC": "Eastern Cape", "FS": "Free State", "GT": "Gauteng", "KZN": "KwaZulu-Natal", "LP": "Limpopo",
         "MP": "Mpumalanga", "NC": "Northern Cape", "NW": "North West", "WC": "Western Cape"}
names = F.create_map(*[F.lit(x) for kv in NAMES.items() for x in kv])
clean_muni = clean_muni.withColumn("province_name", names[F.col("province_code")]).drop("province_name_raw")
clean_muni.write.mode("overwrite").format("delta").saveAsTable("clean_municipalities")
display(clean_muni.groupBy("province_code", "province_name").count().orderBy("province_code"))

## 2. Audit opinions: add an ordinal severity rank

In [ ]:
# 1 = clean ... 5 = disclaimer; outstanding (audit not finalised) kept as its own flag, rank null
RANK = {"unqualified": 1, "unqualified_emphasis_of_matter": 2, "qualified": 3, "adverse": 4, "disclaimer": 5}
rank = F.create_map(*[F.lit(x) for kv in RANK.items() for x in kv])

clean_audit = (raw["audit_opinions"]
    .select(F.col("demarcation_code"), F.col("financial_year_end_year").alias("year"),
            F.col("opinion_code"), F.col("opinion_label"), F.col("opinion_report_url").alias("report_url"))
    .withColumn("opinion_rank", rank[F.col("opinion_code")])
    .withColumn("is_outstanding", F.col("opinion_code") == "outstanding")
    .withColumn("is_clean", F.col("opinion_code") == "unqualified"))
clean_audit.write.mode("overwrite").format("delta").saveAsTable("clean_audit_opinions")

## 3. Irregular, fruitless and unauthorised expenditure: blanks → 0 with a flag, one row per municipality-year

In [ ]:
u = (raw["uifwexp"]
     .select("demarcation_code", F.col("financial_year_end_year").alias("year"), F.col("item_code").alias("item"), "amount")
     .withColumn("was_blank", F.col("amount").isNull())
     .withColumn("amount", F.coalesce(F.col("amount"), F.lit(0.0))))

clean_uifw = (u.groupBy("demarcation_code", "year")
    .pivot("item", ["irregular", "fruitless", "unauthorised"])
    .agg(F.first("amount").alias("amt"), F.first("was_blank").alias("was_blank")))
clean_uifw = clean_uifw.toDF(*[c.replace("_amt", "_exp").replace("_was_blank", "_was_blank") for c in clean_uifw.columns])
clean_uifw = clean_uifw.withColumn("uifw_total", F.col("irregular_exp") + F.col("fruitless_exp") + F.col("unauthorised_exp"))
clean_uifw.write.mode("overwrite").format("delta").saveAsTable("clean_uifw")
clean_uifw.printSchema()

## 4. Audited financial figures: keep every amount type, tidy names, flag availability

In [ ]:
def tidy(df, value_cols):
    out = df.select(F.col("demarcation_code"), F.col("financial_year_end_year").alias("year"),
                    F.col("item_code"), F.col("item_label"), F.col("amount_type_code").alias("amount_type"),
                    *[F.col(c).alias(c.replace("_sum", "")) for c in value_cols])
    return out

clean_incexp = tidy(raw["incexp_v2"], ["amount_sum"])
clean_fpos   = tidy(raw["financial_position_v2"], ["amount_sum"])
clean_debt   = tidy(raw["aged_debtor_v2"], [c for c in raw["aged_debtor_v2"].columns if c.endswith("_sum")])
for name, df in [("clean_incexp", clean_incexp), ("clean_financial_position", clean_fpos), ("clean_aged_debtor", clean_debt)]:
    df.write.mode("overwrite").format("delta").saveAsTable(name)
    print(name, f"{df.count():,} rows")

def audited_flag(df, col):
    return df.where(F.col("amount_type") == "AUDA").select("demarcation_code", "year").distinct().withColumn(col, F.lit(True))

## 5. The panel: one row per municipality per year with an audit opinion

In [ ]:
panel = (clean_audit
    .join(clean_muni.select("demarcation_code", "name", "province_code", "category"), "demarcation_code", "left")
    .join(clean_uifw, ["demarcation_code", "year"], "left")
    .join(audited_flag(clean_incexp, "has_audited_incexp"), ["demarcation_code", "year"], "left")
    .join(audited_flag(clean_fpos, "has_audited_fin_position"), ["demarcation_code", "year"], "left")
    .join(audited_flag(clean_debt, "has_audited_debtors"), ["demarcation_code", "year"], "left")
    .withColumn("mscoa_era", F.col("year") >= 2018)
    # flags are only meaningful where the source table exists: incexp/fin position from 2018, debtors from 2020
    .withColumn("has_audited_incexp", F.when(F.col("year") >= 2018, F.coalesce("has_audited_incexp", F.lit(False))))
    .withColumn("has_audited_fin_position", F.when(F.col("year") >= 2018, F.coalesce("has_audited_fin_position", F.lit(False))))
    .withColumn("has_audited_debtors", F.when(F.col("year") >= 2020, F.coalesce("has_audited_debtors", F.lit(False)))))
panel.write.mode("overwrite").format("delta").saveAsTable("clean_muni_year_panel")
print(f"clean_muni_year_panel: {panel.count():,} rows, {len(panel.columns)} columns")

## 6. Reconciliation checks

In [ ]:
def check(name, ok, detail=""):
    print(("PASS " if ok else "FAIL ") + name + (f"  ({detail})" if detail else ""))

p = spark.table("clean_muni_year_panel")
n_raw = raw["audit_opinions"].count()
check("panel has one row per audit opinion", p.count() == n_raw, f"{p.count()} vs {n_raw}")
check("no duplicate municipality-years", p.groupBy("demarcation_code", "year").count().where("count > 1").count() == 0)
provs = [r[0] for r in spark.table("clean_municipalities").select("province_code").distinct().collect()]
check("9 province codes", len(provs) == 9, sorted(provs))
blanks = raw["uifwexp"].where(F.col("amount").isNull()).count()
flagged = spark.table("clean_uifw").select(sum(F.col(c).cast("int") for c in
          ["irregular_was_blank", "fruitless_was_blank", "unauthorised_was_blank"]).alias("n")).agg(F.sum("n")).first()[0]
check("every blank expenditure amount is flagged", blanks == flagged, f"{blanks} blanks, {flagged} flagged")
raw_total = raw["uifwexp"].agg(F.sum("amount")).first()[0]
clean_total = spark.table("clean_uifw").agg(F.sum("uifw_total")).first()[0]
check("expenditure total unchanged by recoding", abs(raw_total - clean_total) < 1, f"R{raw_total:,.0f}")
miss = p.where("year BETWEEN 2018 AND 2024 AND NOT has_audited_incexp").count()
check("112 municipality-years 2018-2024 without audited income statement", miss == 112, miss)
check("no rows dropped from financial tables",
      spark.table("clean_incexp").count() == raw["incexp_v2"].count()
      and spark.table("clean_financial_position").count() == raw["financial_position_v2"].count()
      and spark.table("clean_aged_debtor").count() == raw["aged_debtor_v2"].count())